In [1]:
# ===============================
# TRY 2 ROBUSTNESS EXPERIMENTS
# ===============================

import json, os, pickle, random, warnings
import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"

OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

print("Robustness notebook started")

Robustness notebook started


In [2]:
with open(f"{BASE}/cps-config/config.json") as f:
    CFG = json.load(f)

TRAIN_IDS = CFG["engine_partition"]["train"]
VAL_IDS   = CFG["engine_partition"]["validation"]

print("Train engines:", len(TRAIN_IDS))
print("Validation engines:", len(VAL_IDS))

Train engines: 70
Validation engines: 15


In [3]:
df = pd.read_csv(
    f"{BASE}/v2-phase3-features/v2_phase3_feature_dataset.csv"
)

with open(f"{BASE}/v2-phase3-features/v2_feature_cols.json") as f:
    feature_cols = json.load(f)

scaler = pickle.load(
    open(f"{BASE}/v2-phase3-features/v2_scaler.pkl","rb")
)

print("Dataset shape:", df.shape)
print("Feature count:", len(feature_cols))

assert "max_cycle" not in feature_cols

Dataset shape: (20631, 65)
Feature count: 56


In [4]:
val_df = df[df.engine_id.isin(VAL_IDS)].copy()

print("Validation rows:", val_df.shape)

Validation rows: (3210, 65)


In [5]:
best_model = pickle.load(
    open(f"{BASE}/v2-baseline-results/v2_best_centralized.pkl","rb")
)

print("Centralized model loaded")

Centralized model loaded


In [6]:
weights_df = pd.read_csv(
    f"{BASE}/v2-node-predictions/v2_node_f1_weights.csv"
)

node_weights = dict(zip(weights_df.node_id, weights_df.weight))

node_models = {}

for nid in node_weights:

    model_path = f"{BASE}/v2-node-predictions/v2_{nid}_xgb.pkl"

    node_models[nid] = pickle.load(open(model_path,"rb"))

print("Node models loaded:", list(node_models.keys()))

Node models loaded: ['node_A', 'node_B', 'node_C', 'node_D', 'node_E']


In [7]:
top_models = pd.read_csv(
    f"{BASE}/v2-baseline-results/v2_top_models.csv"
)

best_window = int(top_models.iloc[0]["window"])

LABEL_COL = f"label_w{best_window}"

print("Using label:", LABEL_COL)

X_val = scaler.transform(val_df[feature_cols])

y_true = val_df[LABEL_COL].values

Using label: label_w20


In [8]:
def distributed_predict(X):

    votes = []

    for nid, model in node_models.items():

        probs = model.predict_proba(X)[:,1]

        flags = (probs > 0.5).astype(int)

        weight = node_weights[nid]

        votes.append(flags * weight)

    weighted = sum(votes)

    return (weighted > 0.5).astype(int)

In [9]:
noise_rows = []

sensor_cols = [c for c in val_df.columns if c.startswith("s") and "_" not in c]

for sigma in [0.1,0.2,0.3]:

    noisy = val_df.copy()

    for s in sensor_cols:
        noisy[s] += np.random.normal(0, sigma, len(noisy))

    X = scaler.transform(noisy[feature_cols])

    c_pred = best_model.predict(X)

    d_pred = distributed_predict(X)

    for sys,preds in [("centralized",c_pred),("distributed",d_pred)]:

        noise_rows.append({
            "experiment":"sensor_noise",
            "level":sigma,
            "system":sys,
            "precision":precision_score(y_true,preds),
            "recall":recall_score(y_true,preds),
            "f1":f1_score(y_true,preds)
        })

    print("Noise level",sigma,"done")

Noise level 0.1 done
Noise level 0.2 done
Noise level 0.3 done


In [10]:
missing_rows = []

for rate in [0.05,0.1,0.2]:

    miss = val_df.copy()

    mask = np.random.rand(len(miss),len(sensor_cols)) < rate

    means = val_df[sensor_cols].mean()

    for i,s in enumerate(sensor_cols):

        miss.loc[mask[:,i],s] = means[s]

    X = scaler.transform(miss[feature_cols])

    c_pred = best_model.predict(X)

    d_pred = distributed_predict(X)

    for sys,preds in [("centralized",c_pred),("distributed",d_pred)]:

        missing_rows.append({
            "experiment":"missing_values",
            "level":rate,
            "system":sys,
            "precision":precision_score(y_true,preds),
            "recall":recall_score(y_true,preds),
            "f1":f1_score(y_true,preds)
        })

    print("Missing rate",rate,"done")

Missing rate 0.05 done
Missing rate 0.1 done
Missing rate 0.2 done


In [11]:
failure_rows = []

for level in [0.1,0.2,0.3]:

    fail = val_df.copy()

    n_drop = max(1,int(level*len(sensor_cols)))

    drop = np.random.choice(sensor_cols,n_drop,replace=False)

    fail[drop] = 0

    X = scaler.transform(fail[feature_cols])

    c_pred = best_model.predict(X)

    d_pred = distributed_predict(X)

    for sys,preds in [("centralized",c_pred),("distributed",d_pred)]:

        failure_rows.append({
            "experiment":"sensor_failure",
            "level":level,
            "system":sys,
            "precision":precision_score(y_true,preds),
            "recall":recall_score(y_true,preds),
            "f1":f1_score(y_true,preds)
        })

    print("Sensor failure",level,"done")

Sensor failure 0.1 done
Sensor failure 0.2 done
Sensor failure 0.3 done


In [12]:
drop_rows = []

nodes = list(node_models.keys())

for rate in [0.1,0.2,0.3]:

    dropped = np.random.choice(nodes, max(1,int(rate*len(nodes))), replace=False)

    votes = []

    for nid, model in node_models.items():

        if nid in dropped:

            flags = np.zeros(len(X_val))
            weight = 0

        else:

            probs = model.predict_proba(X_val)[:,1]

            flags = (probs > 0.5).astype(int)

            weight = node_weights[nid]

        votes.append(flags * weight)

    weighted = sum(votes)

    d_pred = (weighted > 0.5).astype(int)

    noise = X_val + np.random.normal(0,rate,X_val.shape)

    c_pred = best_model.predict(noise)

    for sys,preds in [("centralized",c_pred),("distributed",d_pred)]:

        drop_rows.append({
            "experiment":"node_dropout",
            "level":rate,
            "system":sys,
            "precision":precision_score(y_true,preds),
            "recall":recall_score(y_true,preds),
            "f1":f1_score(y_true,preds)
        })

    print("Node dropout",rate,"done")

Node dropout 0.1 done
Node dropout 0.2 done
Node dropout 0.3 done


In [13]:
rows = noise_rows + missing_rows + failure_rows + drop_rows

robust_df = pd.DataFrame(rows)

dist = robust_df[robust_df.system=="distributed"]

cent = robust_df[robust_df.system=="centralized"]

dist.to_csv(f"{OUT}v2_robustness_distributed.csv",index=False)
cent.to_csv(f"{OUT}v2_robustness_centralized.csv",index=False)

delta = dist.merge(
    cent,
    on=["experiment","level"],
    suffixes=("_dist","_cent")
)

delta["f1_delta"] = delta["f1_dist"] - delta["f1_cent"]

delta.to_csv(f"{OUT}v2_robustness_delta.csv",index=False)

delta[["experiment","level","f1_dist","f1_cent","f1_delta"]]

,experiment,level,f1_dist,f1_cent,f1_delta
0,sensor_noise,0.10,0.854573,0.861897,-0.007324
1,sensor_noise,0.20,0.856716,0.825784,0.030932
2,sensor_noise,0.30,0.847761,0.835664,0.012097
3,missing_values,0.05,0.855440,0.854812,0.000627
4,missing_values,0.10,0.856716,0.842282,0.014435
5,missing_values,0.20,0.857143,0.797178,0.059965
6,sensor_failure,0.10,0.855882,0.860800,-0.004918
7,sensor_failure,0.20,0.856721,0.842105,0.014616
8,sensor_failure,0.30,0.848665,0.679089,0.169576
9,node_dropout,0.10,0.861912,0.503529,0.358383


In [14]:
!ls -lh /kaggle/working/

total 40K
---------- 1 root root  26K Apr 12 14:14 __notebook__.ipynb
-rw-r--r-- 1 root root 1017 Apr 12 14:14 v2_robustness_centralized.csv
-rw-r--r-- 1 root root 2.2K Apr 12 14:14 v2_robustness_delta.csv
-rw-r--r-- 1 root root 1.1K Apr 12 14:14 v2_robustness_distributed.csv
